## Time and Operational Data Preparation
This notebook focuses on preparing the time and operational feature group for the shipment delay prediction project. The goal is to clean and transform the dataset, engineer relevant time-based and operational features, and prepare the data for integration into the final modeling dataset.


##  Import Required Libraries

In this step, we import the necessary Python libraries for data manipulation, analysis, and visualization. These include pandas and numpy for data processing, matplotlib and seaborn for visualization, and s3fs for accessing data stored in Amazon S3.

In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
import s3fs

## Load Data from s3
The raw logistics dataset is loaded from the S3 bucket. This dataset contains shipment-level information used for analysis and modeling.

In [ ]:
df = pd.read_csv("s3://ads508-bereket-bucket/project-raw-data/Logistics_supply_chain.csv")

print(df.shape)
df.head()

## Data Inspection
This step examines the structure of the dataset, checks for missing values, and identifies duplicate records.

In [ ]:
print(df.info())
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

## Convert Timestamp to Datetime Format

The timestamp column is converted into datetime format to enable time-based feature extraction.

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

df["timestamp"].head()

## Create Time-Based Features

Time-based features such as year, month, day, day of the week, and hour are extracted from the timestamp.

In [ ]:
df["year"] = df["timestamp"].dt.year
df["month"] = df["timestamp"].dt.month
df["day"] = df["timestamp"].dt.day
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["hour"] = df["timestamp"].dt.hour

df[["timestamp", "month", "day_of_week", "hour"]].head()

## Create Target Variable

A binary variable `is_delayed` is created based on delivery time deviation.

In [ ]:
df["is_delayed"] = (df["delivery_time_deviation"] > 0).astype(int)

df["is_delayed"].value_counts()

## Create Operational Indicator Features

Binary indicators are created to capture operational risks such as congestion, weather severity, and supplier reliability.

In [ ]:
df["high_port_congestion"] = (df["port_congestion_level"] >= 7).astype(int)
df["bad_weather"] = (df["weather_condition_severity"] >= 0.7).astype(int)
df["low_supplier_reliability"] = (df["supplier_reliability_score"] <= 0.3).astype(int)

df[["high_port_congestion", "bad_weather", "low_supplier_reliability"]].head()

## Select Relevant Features

A subset of relevant columns is selected to create a focused dataset for the time and operational feature group.

In [ ]:
time_operational_df = df[
    [
        "timestamp",
        "lead_time_days",
        "weather_condition_severity",
        "port_congestion_level",
        "supplier_reliability_score",
        "delivery_time_deviation",
        "year",
        "month",
        "day",
        "day_of_week",
        "hour",
        "is_delayed",
        "high_port_congestion",
        "bad_weather",
        "low_supplier_reliability"
    ]
].copy()

time_operational_df.head()

## Validate Prepared Dataset

The dataset is checked again for structure, missing values, and consistency.

In [ ]:
print(time_operational_df.info())
print("\nMissing values:\n", time_operational_df.isnull().sum())

## Save Processed Dataset to S3

The processed dataset is saved to an S3 bucket for integration into the final training dataset.

In [ ]:
output_path = "s3://ads508-bereket-bucket/project-processed-data/bereket_time_operational_features.csv"

time_operational_df.to_csv(output_path, index=False)

print("Saved to:", output_path)

## Save Local Copy

A local copy is saved for sharing with team members and version control.

In [ ]:
time_operational_df.to_csv("bereket_time_operational_features.csv", index=False)

print("Local file saved")

## Summary of Data Preparation

The dataset was successfully cleaned and transformed by converting timestamps, generating time-based features, and creating operational indicators. The processed dataset is now ready for integration into the final modeling pipeline.

## Next Steps

The prepared dataset will be merged with other feature groups to create a unified dataset for training machine learning models to predict shipment delays.